# Session 2, Block 2 &mdash; Data Visualization & EDA
**Data Science Techniques and Real-World Applications &mdash; WS 2026**

Thursday 10 September 2026 &middot; Block 2 (11:15&ndash;12:45)

We'll keep using the tech-stocks data from Block 1, and bring in one classic teaching dataset
(restaurant tips) for variety in the EDA section.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
df.head()

## <span style="color:#1e3a8a">Matplotlib essentials</span>

Matplotlib is the foundation almost every other Python plotting library (including Seaborn) is
built on. Two ways to use it:

- **pyplot (state-machine) API** &mdash; quick, implicit, great for a one-off plot: `plt.plot(...)`.
- **Object-oriented API** &mdash; explicit `fig, ax = plt.subplots()`, then call methods on `ax`.
  More verbose, but the only sane way once you have multiple subplots to manage.

We'll mostly use pyplot for speed, and switch to the OO style when we need more than one axes.


In [ ]:
# pyplot style: quick and implicit
plt.plot([1, 2, 3, 4], [1, 4, 9, 16], "b-o")
plt.title("Quick pyplot example")
plt.show()

In [ ]:
# OO style: explicit figure and axes -- needed once you have multiple subplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot([1, 2, 3, 4], [1, 4, 9, 16])
axes[0].set_title("Left")

axes[1].bar(["A", "B", "C"], [5, 25, 15])
axes[1].set_title("Right")

fig.tight_layout()
plt.show()

### A few common plot types

In [ ]:
aapl = df[df["Ticker Symbol"] == "AAPL"].sort_values("Names Date")

plt.plot(aapl["Names Date"], aapl["Price or Bid/Ask Average"])
plt.title("AAPL price over time")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
plt.hist(df["Returns"], bins=50)
plt.title("Distribution of daily returns, all tickers pooled")
plt.show()

In [ ]:
plt.scatter(df["Returns"], df["Return on the S&P 500 Index"], alpha=0.3, s=8)
plt.xlabel("Stock return")
plt.ylabel("S&P 500 return")
plt.title("Stock returns vs. market returns")
plt.show()

## <span style="color:#1e3a8a">Seaborn essentials</span>

Seaborn wraps Matplotlib with sensible defaults and works directly with tidy DataFrames &mdash;
you name columns instead of pulling out arrays by hand, and grouping (`hue`) comes for free.


In [ ]:
sns.scatterplot(data=df, x="Returns", y="Return on the S&P 500 Index", hue="Ticker Symbol", alpha=0.5)
plt.title("Stock vs. market returns, by ticker")
plt.show()

In [ ]:
sns.histplot(data=df, x="Returns", hue="Ticker Symbol", bins=40, element="step")
plt.title("Return distributions by ticker")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Ticker Symbol", y="Returns")
plt.title("Return spread by ticker")
plt.show()

In [ ]:
sns.barplot(data=df, x="Ticker Symbol", y="Returns", errorbar="sd")
plt.title("Mean daily return by ticker (error bars = 1 std)")
plt.show()

In [ ]:
# a pairplot is a quick way to eyeball every pairwise relationship at once
sample = df[["Ticker Symbol", "Price or Bid/Ask Average", "Returns"]].sample(500, random_state=42)
sns.pairplot(sample, hue="Ticker Symbol", diag_kind="kde")
plt.show()

<span style="color:#b45309">**Exercise 1: Compare volatility visually**</span>

Using `sns.boxplot` or `sns.violinplot`, compare the *spread* of `Returns` across the four tickers. Which ticker looks the most volatile just from the plot -- and does that match what `groups["Returns"].std()` (from Block 1) told you numerically?

Try it in the cell below, then check the answer notebook (`Session 2b - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Exploratory Data Analysis (EDA)</span>

EDA (a term coined by John Tukey) means getting to know a dataset *before* you model or test
anything &mdash; descriptive stats, distributions, correlations, and spotting problems (missing
values, outliers) early. Skipping it is one of the most common causes of an analysis quietly
going wrong.

Let's switch datasets for a moment &mdash; `tips`, a classic small restaurant-bill dataset built
into Seaborn (loaded once and cached locally, so it only needs internet the first time).


In [ ]:
tips = sns.load_dataset("tips")
tips.head()

### Descriptive statistics

In [ ]:
tips.describe()

In [ ]:
tips.describe(include="all")   # mixes numeric + categorical summaries in one table

### Distributions

In [ ]:
sns.histplot(data=tips, x="total_bill", kde=True)
plt.title("Distribution of total bill")
plt.show()

### Correlation

In [ ]:
corr = tips.select_dtypes("number").corr()
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

A heatmap makes correlation structure easy to scan at a glance &mdash; but a correlation, on
its own, is a description of the data you have. It is **not** evidence that one variable causes
another; we'll come back to exactly this distinction properly on Day 3.

### Grouping and aggregation

In [ ]:
tips.groupby("day", observed=True).agg(
    avg_bill=("total_bill", "mean"),
    avg_tip=("tip", "mean"),
    n=("total_bill", "size"),
)

### Missing values and outliers

In [ ]:
tips.isna().sum()

In [ ]:
# a quick, deliberately simple outlier check: values far from the mean in standard-deviation units
z = (tips["total_bill"] - tips["total_bill"].mean()) / tips["total_bill"].std()
tips[z.abs() > 3]

Zero missing values and no extreme outliers here &mdash; `tips` is a clean teaching dataset.
Real data essentially never is; check both, every time, before drawing conclusions.

<span style="color:#b45309">**Exercise 2: EDA on the tech-stocks data**</span>

Apply the same three checks to `df` (the tech-stocks data): (1) a correlation heatmap between `Returns` and `Return on the S&P 500 Index`, (2) `df.isna().sum()`, (3) a simple z-score outlier check on `Returns`. Are there any missing values? Any days that look like outliers?

Try it in the cell below, then check the answer notebook (`Session 2b - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">A note on Auto-EDA tools</span>

Tools like **`sweetviz`**, **`ydata-profiling`**, and **`pandas-profiling`** generate a full
descriptive report &mdash; distributions, correlations, missing-value summaries &mdash; from a
DataFrame in one line. They're a genuinely useful *first pass* on a new dataset.

They are not a substitute for actually looking at your data and thinking about it. An
auto-generated report can't tell you whether a correlation is spurious, whether a variable was
measured sensibly, or whether "no missing values" actually means "missing values were silently
coded as 0." Treat them as a fast starting point, not a finished analysis &mdash; especially since
the whole point of the EDA step above is to build your own judgment about the data, not outsource it.
